# 02 — Downstream bipartite and cross-network analysis

**Public reproducibility version.** This notebook preserves the final analytical logic used for the revised manuscript. Local absolute paths and development-only cells were cleaned for portability. The notebook has not been re-executed as part of repository packaging.


# 02 — Downstream / Manuscript Analysis (NO_BRAIN_FOG)

Run this notebook **after Notebook 01 completes successfully**.

It reads the frozen outputs from:

`04_revision_outputs/main_pipeline_no_brain_fog_final_v1`

and performs all lightweight downstream analyses needed for the revised manuscript:

1. freeze the five reviewed semantic-topic labels,
2. rebuild the corrected user-topic table,
3. distinguish semantic-cluster size from user-participation size,
4. quantify bridging users and topic-topic overlap,
5. reconstruct legacy vs corrected topic coverage,
6. fully characterize corrected no-topic users,
7. draw the manuscript-style Top-5 bipartite figure,
8. classify within/cross-community interactions,
9. create topic × community-relation summaries,
10. add chi-square effect sizes,
11. load and redraw threshold sensitivity results from Notebook 01,
12. run SNA-community × semantic-topic cross-network analyses,
13. apply BH-FDR and report Cohen's w for community-level goodness-of-fit tests,
14. draw a corrected Top-5 + Others adjusted-standardized-residual heatmap.

**No SBERT embeddings, Leiden clustering, or social-network community detection are recomputed here.**


## 0 — Configuration

In [ ]:
from pathlib import Path

import os
import re
from itertools import combinations

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, chisquare
from statsmodels.stats.multitest import multipletests
from sklearn.feature_extraction.text import CountVectorizer
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

UPSTREAM_RUN_NAME = "main_pipeline_no_brain_fog_final_v1"
DOWNSTREAM_RUN_NAME = "downstream_analysis_no_brain_fog_final_v1"

UPSTREAM_DIR = str(PROJECT_ROOT / "outputs" / UPSTREAM_RUN_NAME)
SAVE_DIR = str(PROJECT_ROOT / "outputs" / DOWNSTREAM_RUN_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print("Frozen upstream folder :", UPSTREAM_DIR)
print("Downstream output      :", SAVE_DIR)


## 1 — Load and validate Notebook 01 outputs

In [ ]:

required = {
    "propagated_posts": "filtered_posts_df_2_corrected_propagated.parquet",
    "user_community": "user_community_df.csv",
    "centrality": "centrality_df.csv",
    "semantic_map": "semantic_node_topic_map_revision.csv",
    "threshold_summary": "threshold_sensitivity_summary.csv",
    "mapping_summary": "semantic_mapping_integrity_summary.csv",
    "keyword_audit": "keyword_refinement_no_brain_fog_audit.csv",
}

missing = [
    fname for fname in required.values()
    if not os.path.exists(os.path.join(UPSTREAM_DIR, fname))
]
if missing:
    raise FileNotFoundError(
        "Notebook 01 is incomplete. Missing:\n- " + "\n- ".join(missing)
    )

fp = pd.read_parquet(os.path.join(UPSTREAM_DIR, required["propagated_posts"]))
user_community_df = pd.read_csv(os.path.join(UPSTREAM_DIR, required["user_community"]))
centrality_df = pd.read_csv(os.path.join(UPSTREAM_DIR, required["centrality"]))
semantic_node_topic_map = pd.read_csv(os.path.join(UPSTREAM_DIR, required["semantic_map"]))
threshold_summary = pd.read_csv(os.path.join(UPSTREAM_DIR, required["threshold_summary"]))
mapping_summary = pd.read_csv(os.path.join(UPSTREAM_DIR, required["mapping_summary"]))
keyword_refinement_audit = pd.read_csv(os.path.join(UPSTREAM_DIR, required["keyword_audit"]))

# Normalize critical IDs once.
fp["user_id"] = pd.to_numeric(fp["user_id"], errors="coerce")
fp["partner_id"] = pd.to_numeric(fp["partner_id"], errors="coerce")
fp["topic"] = pd.to_numeric(fp["topic"], errors="coerce").fillna(-1).astype(int)

user_community_df["user_id"] = pd.to_numeric(
    user_community_df["user_id"], errors="coerce"
)
user_community_df = user_community_df.dropna(subset=["user_id"]).copy()
user_community_df["user_id"] = user_community_df["user_id"].astype(int)

print(f"Expanded analytic rows       : {len(fp):,}")
print(f"Rows with semantic topic     : {(fp['topic'] >= 0).sum():,} ({(fp['topic'] >= 0).mean():.1%})")
print(f"LCC users                    : {user_community_df['user_id'].nunique():,}")
print(f"Semantic graph nodes         : {len(semantic_node_topic_map):,}")
print("\nNotebook 01 mapping audit:")
display(mapping_summary)


## 2 — Freeze the five reviewed semantic-topic labels

In [ ]:

semantic_sizes = (
    semantic_node_topic_map["topic"]
    .astype(int)
    .value_counts()
    .sort_values(ascending=False)
)

top5_topic_ids = semantic_sizes.head(5).index.tolist()
EXPECTED_TOP5 = [0, 1, 2, 3, 4]

print("Five largest topic IDs:", top5_topic_ids)
display(semantic_sizes.head(10).rename("n_unique_processed_texts").to_frame())

if top5_topic_ids != EXPECTED_TOP5:
    raise RuntimeError(
        "The Top-5 topics are no longer Topics 0–4. "
        "Stop and re-review the Top-5 content-audit packet from Notebook 01."
    )

TOPIC_LABELS = {
    0: "Caregiving and Family Support",
    1: "Suspicions and Uncertainty",
    2: "Family Loss and Grief",
    3: "Research on Diagnosis and Treatment",
    4: "Risk and Public Health",
}

# Keep the same final manuscript wording everywhere, including figures/heatmaps.
SHORT_TOPIC_LABELS = TOPIC_LABELS.copy()

topic_label_table = pd.DataFrame({
    "topic": top5_topic_ids,
    "topic_label": [TOPIC_LABELS[t] for t in top5_topic_ids],
    "semantic_size": [int(semantic_sizes.loc[t]) for t in top5_topic_ids],
})
display(topic_label_table)
topic_label_table.to_csv(
    os.path.join(SAVE_DIR, "final_top5_topic_labels.csv"),
    index=False
)


## 3 — Corrected user–topic table

In [ ]:

df_pos = fp.loc[fp["topic"] >= 0].copy()

user_topic_from_author = (
    df_pos[["user_id", "topic"]]
    .dropna(subset=["user_id"])
    .rename(columns={"user_id": "user"})
)
user_topic_from_partner = (
    df_pos[["partner_id", "topic"]]
    .dropna(subset=["partner_id"])
    .rename(columns={"partner_id": "user"})
)

user_topic_long = pd.concat(
    [user_topic_from_author, user_topic_from_partner],
    ignore_index=True
).dropna(subset=["user", "topic"])

user_topic_long["user"] = pd.to_numeric(
    user_topic_long["user"], errors="coerce"
)
user_topic_long["topic"] = pd.to_numeric(
    user_topic_long["topic"], errors="coerce"
)
user_topic_long = user_topic_long.dropna(subset=["user", "topic"]).copy()
user_topic_long["user"] = user_topic_long["user"].astype(int)
user_topic_long["topic"] = user_topic_long["topic"].astype(int)

user_topic_long_clean = (
    user_topic_long
    .drop_duplicates(["user", "topic"])
    .reset_index(drop=True)
)

author_topic_long_clean = (
    user_topic_from_author
    .dropna(subset=["user", "topic"])
    .copy()
)
author_topic_long_clean["user"] = author_topic_long_clean["user"].astype(int)
author_topic_long_clean["topic"] = author_topic_long_clean["topic"].astype(int)
author_topic_long_clean = author_topic_long_clean.drop_duplicates(["user", "topic"])

n_users_topic = user_topic_long_clean["user"].nunique()
n_edges_topic = len(user_topic_long_clean)

print("=== Corrected User–Topic Network ===")
print(f"Users (author or partner) : {n_users_topic:,}")
print(f"Author users with topic   : {author_topic_long_clean['user'].nunique():,}")
print(f"Topics represented        : {user_topic_long_clean['topic'].nunique():,}")
print(f"Unique user-topic edges   : {n_edges_topic:,}")


## 4 — Semantic size vs user-participation size

These are deliberately kept separate.

- **Manuscript Top-5 topics:** defined by topic size in Notebook 01.
- **User-participation ranking:** number of unique users linked to each topic.

The second ranking is descriptive only and must not redefine the manuscript Top-5.


In [ ]:

topic_user_counts = (
    user_topic_long_clean
    .groupby("topic")["user"]
    .nunique()
    .sort_values(ascending=False)
)

topic_participation_df = topic_user_counts.rename("n_users").reset_index()
topic_participation_df["semantic_rank"] = (
    topic_participation_df["topic"]
    .map({t: i+1 for i, t in enumerate(semantic_sizes.index.tolist())})
)

print("Top 10 topics by USER participation:")
display(topic_participation_df.head(10))

topic_participation_df.to_csv(
    os.path.join(SAVE_DIR, "topic_size_by_user_participation.csv"),
    index=False
)

true_top5_users = user_topic_long_clean.loc[
    user_topic_long_clean["topic"].isin(top5_topic_ids), "user"
].nunique()

print(
    f"Users linked to at least one of the five largest topics: "
    f"{true_top5_users:,} ({true_top5_users/n_users_topic:.2%} of topic-linked users)"
)


## 5 — Participation breadth and bridging users

In [ ]:

user_topic_counts = (
    user_topic_long_clean
    .groupby("user")["topic"]
    .nunique()
)

bridging_summary = pd.DataFrame([{
    "topic_linked_users": len(user_topic_counts),
    "users_ge_2_topics": int((user_topic_counts >= 2).sum()),
    "pct_ge_2_topics": float((user_topic_counts >= 2).mean()),
    "users_ge_3_topics": int((user_topic_counts >= 3).sum()),
    "pct_ge_3_topics": float((user_topic_counts >= 3).mean()),
    "median_topics_per_user": float(user_topic_counts.median()),
    "mean_topics_per_user": float(user_topic_counts.mean()),
}])

display(
    bridging_summary.style.format({
        "pct_ge_2_topics": "{:.2%}",
        "pct_ge_3_topics": "{:.2%}",
        "mean_topics_per_user": "{:.3f}",
    })
)

bridging_summary.to_csv(
    os.path.join(SAVE_DIR, "user_topic_bridging_summary.csv"),
    index=False
)

user_topic_counts.rename("n_topics").reset_index().to_csv(
    os.path.join(SAVE_DIR, "user_topic_participation_breadth.csv"),
    index=False
)


## 6 — Topic–topic user overlap

In [ ]:

topic_user_sets = {
    int(t): set(grp["user"].astype(int))
    for t, grp in user_topic_long_clean.groupby("topic")
}

overlap_records = []
for t1, t2 in combinations(sorted(topic_user_sets), 2):
    shared = topic_user_sets[t1] & topic_user_sets[t2]
    overlap_records.append({
        "topic_1": t1,
        "topic_2": t2,
        "shared_users": len(shared),
        "pct_of_topic_1_users": (
            len(shared) / len(topic_user_sets[t1]) if topic_user_sets[t1] else 0
        ),
        "pct_of_topic_2_users": (
            len(shared) / len(topic_user_sets[t2]) if topic_user_sets[t2] else 0
        ),
    })

overlap_df = (
    pd.DataFrame(overlap_records)
    .sort_values("shared_users", ascending=False)
)

display(overlap_df.head(20))
overlap_df.to_csv(
    os.path.join(SAVE_DIR, "topic_topic_user_overlap.csv"),
    index=False
)


## 7 — Legacy vs corrected user-level topic coverage

In [ ]:

all_users = set(
    pd.concat([fp["user_id"], fp["partner_id"]])
    .dropna()
    .astype(int)
    .unique()
)

# Reconstruct the old incorrect mapping:
# semantic_node_id was mistakenly treated as expanded-row original_index.
legacy_post_topic = (
    semantic_node_topic_map[["semantic_node_id", "topic"]]
    .rename(columns={"semantic_node_id": "original_index"})
    .copy()
)
legacy_post_topic["original_index"] = legacy_post_topic["original_index"].astype(int)

legacy_fp = fp.drop(columns=["topic"]).merge(
    legacy_post_topic,
    on="original_index",
    how="left",
    validate="one_to_one",
    sort=False
)
legacy_fp["topic"] = legacy_fp["topic"].fillna(-1).astype(int)

legacy_pos = legacy_fp.loc[legacy_fp["topic"] >= 0]
legacy_user_topic = pd.concat([
    legacy_pos[["user_id", "topic"]]
        .dropna(subset=["user_id"])
        .rename(columns={"user_id": "user"}),
    legacy_pos[["partner_id", "topic"]]
        .dropna(subset=["partner_id"])
        .rename(columns={"partner_id": "user"}),
], ignore_index=True).dropna()

legacy_user_topic["user"] = legacy_user_topic["user"].astype(int)
legacy_user_topic["topic"] = legacy_user_topic["topic"].astype(int)
legacy_user_topic = legacy_user_topic.drop_duplicates(["user", "topic"])

legacy_topic_users = set(legacy_user_topic["user"])
corrected_topic_users = set(user_topic_long_clean["user"])

lcc_users = set(user_community_df["user_id"].astype(int))

coverage_summary = pd.DataFrame([
    {
        "definition": "legacy_mapping_all_users",
        "denominator_users": len(all_users),
        "users_with_topic": len(all_users & legacy_topic_users),
        "users_without_topic": len(all_users - legacy_topic_users),
        "no_topic_pct": len(all_users - legacy_topic_users) / len(all_users),
    },
    {
        "definition": "corrected_mapping_all_users",
        "denominator_users": len(all_users),
        "users_with_topic": len(all_users & corrected_topic_users),
        "users_without_topic": len(all_users - corrected_topic_users),
        "no_topic_pct": len(all_users - corrected_topic_users) / len(all_users),
    },
    {
        "definition": "corrected_mapping_LCC_only",
        "denominator_users": len(lcc_users),
        "users_with_topic": len(lcc_users & corrected_topic_users),
        "users_without_topic": len(lcc_users - corrected_topic_users),
        "no_topic_pct": len(lcc_users - corrected_topic_users) / len(lcc_users),
    },
])

display(
    coverage_summary.style.format({"no_topic_pct": "{:.2%}"})
)

coverage_summary.to_csv(
    os.path.join(SAVE_DIR, "user_topic_coverage_legacy_vs_corrected.csv"),
    index=False
)


## 8 — Full corrected no-topic user audit

In [ ]:

# Final 17-keyword specification, used only for descriptive keyword comparisons.
ORIGINAL_KEYWORDS_20 = [
    "Alzheimer", "dementia", "ADRD",
    "brain fog", "cognitive decline", "forgetfulness", "cognitive test",
    "memory loss", "apoe", "amyloid", "Aducanumab", "Donanemab",
    "Lecanemab", "Donepezil", "Memantine", "Risperidone", "foggy brain",
    "cognitive fog", "mild cognitive", "cognitive training"
]
REMOVED_NONSPECIFIC_KEYWORDS = [
    "brain fog", "foggy brain", "cognitive fog"
]
keywords = [
    kw for kw in ORIGINAL_KEYWORDS_20
    if kw.lower() not in {x.lower() for x in REMOVED_NONSPECIFIC_KEYWORDS}
]
exact_match_lower = {"adrd", "apoe"}

audit_cols = [
    "original_index", "user_id", "partner_id", "text", "text_processed",
    "interaction_type", "has_url", "has_hashtag", "topic"
]
missing_audit_cols = [c for c in audit_cols if c not in fp.columns]
if missing_audit_cols:
    raise KeyError(
        "Propagated parquet lacks required no-topic audit columns: "
        + ", ".join(missing_audit_cols)
    )

audit_posts = fp[audit_cols].copy()

audit_posts["processed_nonempty"] = (
    audit_posts["text_processed"].notna()
    & audit_posts["text_processed"].astype(str).str.strip().ne("")
)

audit_posts["semantic_status"] = np.select(
    [
        ~audit_posts["processed_nonempty"],
        audit_posts["topic"] >= 0,
    ],
    [
        "empty_after_preprocessing",
        "assigned_semantic_topic",
    ],
    default="isolated_below_similarity_threshold"
)

print("=== Post-Level Semantic Status ===")
display(
    audit_posts["semantic_status"]
    .value_counts()
    .rename("n_rows")
    .to_frame()
)

all_user_index = pd.Index(
    pd.concat([audit_posts["user_id"], audit_posts["partner_id"]])
    .dropna()
    .astype(int)
    .unique(),
    name="user"
)
user_audit = pd.DataFrame(index=all_user_index).reset_index()

author_stats = (
    audit_posts.dropna(subset=["user_id"])
    .groupby("user_id")
    .agg(
        n_authored_rows=("original_index", "size"),
        n_unique_authored_texts=("text_processed", "nunique"),
        n_topic_rows_authored=("topic", lambda s: int((s >= 0).sum())),
        n_isolate_rows_authored=(
            "semantic_status",
            lambda s: int((s == "isolated_below_similarity_threshold").sum())
        ),
        n_empty_rows_authored=(
            "semantic_status",
            lambda s: int((s == "empty_after_preprocessing").sum())
        ),
        mean_has_url=("has_url", "mean"),
        mean_has_hashtag=("has_hashtag", "mean"),
    )
    .reset_index()
    .rename(columns={"user_id": "user"})
)
author_stats["user"] = author_stats["user"].astype(int)

partner_stats = (
    audit_posts.dropna(subset=["partner_id"])
    .groupby("partner_id")
    .agg(n_partner_rows=("original_index", "size"))
    .reset_index()
    .rename(columns={"partner_id": "user"})
)
partner_stats["user"] = partner_stats["user"].astype(int)

user_audit = (
    user_audit
    .merge(author_stats, on="user", how="left")
    .merge(partner_stats, on="user", how="left")
)

count_cols = [
    "n_authored_rows", "n_unique_authored_texts", "n_topic_rows_authored",
    "n_isolate_rows_authored", "n_empty_rows_authored", "n_partner_rows"
]
for col in count_cols:
    user_audit[col] = user_audit[col].fillna(0).astype(int)

for col in ["mean_has_url", "mean_has_hashtag"]:
    user_audit[col] = user_audit[col].fillna(0.0)

topic_users_any = set(user_topic_long_clean["user"].astype(int))
topic_users_author = set(author_topic_long_clean["user"].astype(int))

user_audit["has_topic_any_participation"] = user_audit["user"].isin(topic_users_any)
user_audit["has_topic_as_author"] = user_audit["user"].isin(topic_users_author)
user_audit["is_author"] = user_audit["n_authored_rows"] > 0
user_audit["is_partner_only"] = (
    (user_audit["n_authored_rows"] == 0)
    & (user_audit["n_partner_rows"] > 0)
)
user_audit["in_lcc"] = user_audit["user"].isin(lcc_users)
user_audit["topic_status"] = np.where(
    user_audit["has_topic_any_participation"],
    "topic_user",
    "no_topic_user"
)

def assign_no_topic_reason(row):
    if row["has_topic_any_participation"]:
        return "has_topic"
    if row["n_authored_rows"] == 0:
        return "partner_only_and_no_topic_associated_interaction"
    if row["n_topic_rows_authored"] > 0:
        return "unexpected_mapping_inconsistency"
    if row["n_empty_rows_authored"] == row["n_authored_rows"]:
        return "all_authored_rows_empty_after_preprocessing"
    if row["n_isolate_rows_authored"] > 0:
        return "all_nonempty_authored_texts_isolated_below_threshold"
    return "other_no_topic_reason"

user_audit["no_topic_reason"] = user_audit.apply(assign_no_topic_reason, axis=1)

no_topic_users_df = user_audit.loc[
    ~user_audit["has_topic_any_participation"]
].copy()

reason_summary = (
    no_topic_users_df["no_topic_reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="n_users")
)
reason_summary["pct_of_no_topic_users"] = (
    reason_summary["n_users"] / len(no_topic_users_df)
)

print("=== Corrected No-Topic Reasons ===")
display(
    reason_summary.style.format({"pct_of_no_topic_users": "{:.2%}"})
)

def summarize_group(df, label):
    return {
        "group": label,
        "n_users": len(df),
        "pct_in_lcc": df["in_lcc"].mean(),
        "pct_author": df["is_author"].mean(),
        "pct_partner_only": df["is_partner_only"].mean(),
        "median_authored_rows": df["n_authored_rows"].median(),
        "mean_authored_rows": df["n_authored_rows"].mean(),
        "median_unique_authored_texts": df["n_unique_authored_texts"].median(),
        "median_partner_rows": df["n_partner_rows"].median(),
        "mean_url_fraction_among_authored_rows": df["mean_has_url"].mean(),
        "mean_hashtag_fraction_among_authored_rows": df["mean_has_hashtag"].mean(),
    }

group_summary = pd.DataFrame([
    summarize_group(
        user_audit.loc[user_audit["has_topic_any_participation"]],
        "topic_user"
    ),
    summarize_group(
        user_audit.loc[~user_audit["has_topic_any_participation"]],
        "no_topic_user"
    ),
])

print("=== Topic vs No-Topic User Characteristics ===")
display(group_summary)

no_topic_user_set = set(no_topic_users_df["user"].astype(int))
topic_user_set = set(
    user_audit.loc[user_audit["has_topic_any_participation"], "user"].astype(int)
)

authored_posts = audit_posts.copy()
authored_posts["author_group"] = np.select(
    [
        authored_posts["user_id"].fillna(-1).astype(int).isin(no_topic_user_set),
        authored_posts["user_id"].fillna(-1).astype(int).isin(topic_user_set),
    ],
    ["no_topic_user", "topic_user"],
    default="other"
)

interaction_dist = (
    authored_posts.loc[
        authored_posts["author_group"].isin(["topic_user", "no_topic_user"])
    ]
    .groupby(["author_group", "interaction_type"])
    .size()
    .reset_index(name="n_rows")
)
interaction_dist["proportion_within_group"] = (
    interaction_dist["n_rows"]
    / interaction_dist.groupby("author_group")["n_rows"].transform("sum")
)

keyword_records = []
for group_name, grp in authored_posts.loc[
    authored_posts["author_group"].isin(["topic_user", "no_topic_user"])
].groupby("author_group"):
    for kw in keywords:
        pattern = (
            rf"\b{re.escape(kw)}\b"
            if kw.lower() in exact_match_lower
            else rf"{re.escape(kw)}"
        )
        hits = grp["text"].astype(str).str.contains(
            pattern, case=False, regex=True, na=False
        )
        keyword_records.append({
            "author_group": group_name,
            "keyword": kw,
            "matched_rows": int(hits.sum()),
            "total_rows": len(grp),
            "proportion": float(hits.mean()),
        })

keyword_comparison = pd.DataFrame(keyword_records)

def top_terms_for_group(text_series, group_name, top_n=40):
    texts = text_series.dropna().astype(str).str.strip()
    texts = texts[texts.ne("")]
    if len(texts) == 0:
        return pd.DataFrame(columns=["author_group", "term", "count"])

    vectorizer = CountVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_features=5000
    )
    X = vectorizer.fit_transform(texts)
    counts = np.asarray(X.sum(axis=0)).ravel()
    terms = np.asarray(vectorizer.get_feature_names_out())
    order = np.argsort(counts)[::-1][:top_n]
    return pd.DataFrame({
        "author_group": group_name,
        "term": terms[order],
        "count": counts[order].astype(int),
    })

top_terms_comparison = pd.concat([
    top_terms_for_group(
        authored_posts.loc[
            authored_posts["author_group"] == group_name, "text_processed"
        ],
        group_name
    )
    for group_name in ["topic_user", "no_topic_user"]
], ignore_index=True)

def matched_keyword_list(text):
    hits = []
    for kw in keywords:
        pattern = (
            rf"\b{re.escape(kw)}\b"
            if kw.lower() in exact_match_lower
            else rf"{re.escape(kw)}"
        )
        if re.search(pattern, str(text), flags=re.IGNORECASE):
            hits.append(kw)
    return "; ".join(hits)

sample_pool = authored_posts.loc[
    authored_posts["user_id"].fillna(-1).astype(int).isin(no_topic_user_set)
].copy()
sample_pool["matched_keywords"] = sample_pool["text"].apply(matched_keyword_list)
sample_pool = sample_pool.drop_duplicates(subset=["text"])

sample_n = min(100, len(sample_pool))
no_topic_post_sample = (
    sample_pool.sample(n=sample_n, random_state=42)[
        [
            "text", "interaction_type", "semantic_status",
            "matched_keywords", "has_url", "has_hashtag"
        ]
    ]
    .reset_index(drop=True)
)

# Save all no-topic audit outputs.
user_audit.to_csv(
    os.path.join(SAVE_DIR, "corrected_user_topic_status_all_users.csv"),
    index=False
)
no_topic_users_df.to_csv(
    os.path.join(SAVE_DIR, "corrected_no_topic_users_audit.csv"),
    index=False
)
reason_summary.to_csv(
    os.path.join(SAVE_DIR, "corrected_no_topic_reason_summary.csv"),
    index=False
)
group_summary.to_csv(
    os.path.join(SAVE_DIR, "topic_vs_no_topic_user_characteristics.csv"),
    index=False
)
interaction_dist.to_csv(
    os.path.join(SAVE_DIR, "topic_vs_no_topic_interaction_distribution.csv"),
    index=False
)
keyword_comparison.to_csv(
    os.path.join(SAVE_DIR, "topic_vs_no_topic_keyword_distribution.csv"),
    index=False
)
top_terms_comparison.to_csv(
    os.path.join(SAVE_DIR, "topic_vs_no_topic_top_terms.csv"),
    index=False
)
no_topic_post_sample.to_csv(
    os.path.join(SAVE_DIR, "no_topic_post_sample_for_manual_review.csv"),
    index=False
)

print("Saved complete no-topic audit.")


## 9 — Manuscript-style Top-5 user–topic bipartite figure

In [ ]:

# Preserve the original manuscript figure structure:
# spring layout, users as circles, topics as larger squares.
# Use the manuscript Top-5 topics = Topics 0–4.

plot_df = (
    user_topic_long
    .loc[user_topic_long["topic"].isin(top5_topic_ids)]
    .groupby(["user", "topic"])
    .size()
    .reset_index(name="weight")
)

BIPARTITE_USER_FRAC = 0.50

sampled_users = (
    plot_df["user"]
    .drop_duplicates()
    .sample(frac=BIPARTITE_USER_FRAC, random_state=42)
)
sampled_df = plot_df.loc[
    plot_df["user"].isin(sampled_users)
].copy()

B_vis = nx.Graph()
for _, row in sampled_df.iterrows():
    B_vis.add_edge(
        f"U_{int(row['user'])}",
        f"T_{int(row['topic'])}",
        weight=float(row["weight"])
    )

pos = nx.spring_layout(
    B_vis,
    seed=42,
    k=2.0,
    iterations=300,
    weight="weight"
)

users_vis = [n for n in B_vis.nodes if n.startswith("U_")]
topics_vis = [
    f"T_{t}" for t in top5_topic_ids
    if f"T_{t}" in B_vis.nodes
]

topic_color_map = {
    "T_0": "#0B1F5B",
    "T_1": "#1F8ED6",
    "T_2": "#1E6F3D",
    "T_3": "#2ECC71",
    "T_4": "#C0392B",
}

fig, ax = plt.subplots(figsize=(13, 10))

edge_w = np.array(
    [B_vis[u][v]["weight"] for u, v in B_vis.edges()],
    dtype=float
)

nx.draw_networkx_edges(
    B_vis, pos,
    alpha=0.08,
    width=0.5 + 0.12 * edge_w,
    edge_color="0.40",
    ax=ax
)

user_deg = np.array(
    [B_vis.degree(u) for u in users_vis],
    dtype=float
)
if len(user_deg) and user_deg.max() > 0:
    user_sizes = 22 + 95 * (user_deg / user_deg.max())
else:
    user_sizes = np.full(len(users_vis), 22.0)

nx.draw_networkx_nodes(
    B_vis, pos,
    nodelist=users_vis,
    node_color="#4C72B0",
    node_size=user_sizes,
    alpha=0.82,
    edgecolors="white",
    linewidths=0.3,
    ax=ax
)

for tnode in topics_vis:
    nx.draw_networkx_nodes(
        B_vis, pos,
        nodelist=[tnode],
        node_shape="s",
        node_color=topic_color_map[tnode],
        node_size=500,
        alpha=0.98,
        edgecolors="black",
        linewidths=1.2,
        ax=ax
    )

legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="#4C72B0",
        markersize=8,
        label="Users"
    )
] + [
    Patch(
        facecolor=topic_color_map[f"T_{t}"],
        edgecolor="black",
        label=TOPIC_LABELS[t]
    )
    for t in top5_topic_ids
]

ax.legend(
    handles=legend_elements,
    frameon=True,
    fontsize=9,
    loc="upper right"
)

ax.set_title(
    "User–Topic Bipartite Network\n"
    "Five Largest Topics, 50% of Linked Users",
    fontsize=14
)
ax.axis("off")
plt.tight_layout()

bipartite_path = os.path.join(
    SAVE_DIR,
    "user_topic_bipartite_top5_ORIGINAL_STYLE_FIXED.png"
)
plt.savefig(
    bipartite_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

print(
    f"Bipartite graph — nodes: {B_vis.number_of_nodes():,}, "
    f"edges: {B_vis.number_of_edges():,}"
)
print("Saved:", bipartite_path)


## 10 — Correct row-level social-community relation

In [ ]:

user_community_map = dict(
    zip(
        user_community_df["user_id"].astype(int),
        user_community_df["community"]
    )
)

posts_with_comm_type = fp.copy()
posts_with_comm_type["community_user"] = (
    posts_with_comm_type["user_id"]
    .map(user_community_map)
)
posts_with_comm_type["community_partner"] = (
    posts_with_comm_type["partner_id"]
    .map(user_community_map)
)

def classify_community_relation(row):
    if pd.isna(row["partner_id"]):
        return "itself"
    if pd.isna(row["community_user"]) or pd.isna(row["community_partner"]):
        return "unknown"
    if row["community_user"] == row["community_partner"]:
        return "community_inner"
    return "community_across"

posts_with_comm_type["community_type"] = posts_with_comm_type.apply(
    classify_community_relation,
    axis=1
)

community_type_counts = (
    posts_with_comm_type["community_type"]
    .value_counts(dropna=False)
    .rename_axis("community_type")
    .reset_index(name="count")
)
community_type_counts["proportion"] = (
    community_type_counts["count"]
    / community_type_counts["count"].sum()
)

display(
    community_type_counts.style.format({"proportion": "{:.2%}"})
)
community_type_counts.to_csv(
    os.path.join(SAVE_DIR, "community_type_counts_row_level_fixed.csv"),
    index=False
)


## 11 — Topic × community-relation summaries

In [ ]:

topic_comm_rows = posts_with_comm_type.loc[
    posts_with_comm_type["topic"] >= 0
].copy()

topic_community_dist = (
    topic_comm_rows
    .groupby(["topic", "community_type"])
    .size()
    .unstack(fill_value=0)
)

topic_community_prop = topic_community_dist.div(
    topic_community_dist.sum(axis=1),
    axis=0
)

top5_topic_community_prop = (
    topic_community_prop
    .reindex(top5_topic_ids)
    .copy()
)
top5_topic_community_prop.insert(
    0,
    "topic_label",
    [TOPIC_LABELS[t] for t in top5_topic_community_prop.index]
)

print("Top-5 topics × row-level community relation:")
display(top5_topic_community_prop)

topic_community_dist.to_csv(
    os.path.join(SAVE_DIR, "topic_by_community_type_counts_fixed.csv")
)
topic_community_prop.to_csv(
    os.path.join(SAVE_DIR, "topic_by_community_type_proportions_fixed.csv")
)
top5_topic_community_prop.to_csv(
    os.path.join(SAVE_DIR, "top5_topic_by_community_type_proportions_labeled.csv")
)


## 12 — Interaction type × community relation: chi-square + Cramér's V

In [ ]:

interaction_test_df = posts_with_comm_type.loc[
    posts_with_comm_type["community_type"].isin(
        ["community_inner", "community_across"]
    )
    & posts_with_comm_type["interaction_type"].notna()
].copy()

interaction_table = pd.crosstab(
    interaction_test_df["interaction_type"],
    interaction_test_df["community_type"]
)

chi2_int, p_int, dof_int, expected_int = chi2_contingency(
    interaction_table
)
n_int = interaction_table.to_numpy().sum()
v_int = np.sqrt(
    chi2_int
    / (n_int * (min(interaction_table.shape) - 1))
)

print("Interaction type × within/cross-community relation")
print(f"Chi² = {chi2_int:.3f}")
print(f"df   = {dof_int}")
print(f"p    = {p_int:.3e}")
print(f"Cramér's V = {v_int:.3f}")
display(interaction_table)

pd.DataFrame([{
    "chi2": chi2_int,
    "df": dof_int,
    "p_value": p_int,
    "cramers_v": v_int,
    "n_rows": n_int,
}]).to_csv(
    os.path.join(
        SAVE_DIR,
        "interaction_type_community_relation_chi2_effect_size.csv"
    ),
    index=False
)


## 13 — Threshold sensitivity: manuscript table and figure

Notebook 01 already performed the expensive 0.60–0.85 semantic-graph/Leiden sweep.
Notebook 02 **does not recompute it**. It loads the frozen CSV and regenerates the
manuscript-ready table and dual-axis figure.


In [ ]:

PAPER_THRESHOLD = 0.70

display_threshold = threshold_summary.copy()
for col in ["node_retention", "top1_share", "top5_share"]:
    if col in display_threshold.columns:
        display_threshold[col] = display_threshold[col].map("{:.1%}".format)

display(display_threshold)

threshold_summary.to_csv(
    os.path.join(SAVE_DIR, "threshold_sensitivity_summary_manuscript.csv"),
    index=False
)

thr_vals = threshold_summary["threshold"].tolist()
q_vals = threshold_summary["modularity"].tolist()
ret_vals = (threshold_summary["node_retention"] * 100).tolist()
paper_idx = thr_vals.index(PAPER_THRESHOLD)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

l1, = ax1.plot(
    thr_vals, q_vals,
    marker="o",
    color="#D7191C",
    linewidth=2.2,
    markersize=7,
    label="Modularity Q"
)
l2, = ax2.plot(
    thr_vals, ret_vals,
    marker="s",
    color="#2C7BB6",
    linewidth=2.2,
    markersize=7,
    linestyle="--",
    label="Node Retention"
)

ax1.scatter(
    [PAPER_THRESHOLD], [q_vals[paper_idx]],
    s=130, color="#D7191C",
    edgecolors="black", linewidths=1.5, zorder=5
)
ax2.scatter(
    [PAPER_THRESHOLD], [ret_vals[paper_idx]],
    s=130, color="#2C7BB6",
    edgecolors="black", linewidths=1.5, zorder=5
)

ax1.annotate(
    f"Q={q_vals[paper_idx]:.3f}",
    xy=(PAPER_THRESHOLD, q_vals[paper_idx]),
    xytext=(6, -14),
    textcoords="offset points",
    fontsize=9
)
ax2.annotate(
    f"{ret_vals[paper_idx]:.1f}%",
    xy=(PAPER_THRESHOLD, ret_vals[paper_idx]),
    xytext=(6, 6),
    textcoords="offset points",
    fontsize=9
)

ax1.axvline(
    PAPER_THRESHOLD,
    color="0.35",
    linestyle=":",
    linewidth=1.5
)

ax1.set_xlabel("Cosine Similarity Threshold")
ax1.set_ylabel("Modularity Q")
ax2.set_ylabel("Node Retention (%)")
ax1.set_xticks(thr_vals)
ax1.grid(axis="y", linestyle="--", alpha=0.35)

ax1.legend(
    handles=[
        l1, l2,
        plt.Line2D(
            [0], [0],
            color="0.35",
            linestyle=":",
            linewidth=1.5,
            label="Selected threshold (0.70)"
        )
    ],
    fontsize=9,
    loc="center left"
)

plt.title(
    "Threshold Sensitivity: Modularity Q vs Node Retention",
    fontweight="bold"
)
plt.tight_layout()

threshold_fig_path = os.path.join(
    SAVE_DIR, "threshold_sensitivity_dualaxis_MANUSCRIPT.png"
)
plt.savefig(
    threshold_fig_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

selected_row = threshold_summary.loc[
    threshold_summary["threshold"] == PAPER_THRESHOLD
].iloc[0]

print("\nSelected threshold summary:")
print(f"Threshold      : {PAPER_THRESHOLD}")
print(f"Node retention : {selected_row['node_retention']:.1%}")
print(f"Nodes          : {int(selected_row['n_nodes']):,}")
print(f"Communities    : {int(selected_row['n_communities'])}")
print(f"Modularity Q   : {selected_row['modularity']:.4f}")
print(f"Top-5 share    : {selected_row['top5_share']:.1%}")


## 14 — Cross-network analysis

For interpretability, the five largest topics are kept separately and
all remaining topics are collapsed into **Others**.

This prevents the old heatmap problem where every non-top-5 topic appeared as a
separate column.


In [ ]:

cross = user_topic_long_clean.copy()
cross["sna_community"] = cross["user"].map(user_community_map)
cross = cross.dropna(subset=["sna_community"]).copy()
cross["sna_community"] = cross["sna_community"].astype(int)

def collapse_topic(t):
    if t in TOPIC_LABELS:
        return SHORT_TOPIC_LABELS[t]
    return "Others"

cross["topic_group"] = cross["topic"].map(collapse_topic)

TOPIC_ORDER = [
    SHORT_TOPIC_LABELS[t] for t in top5_topic_ids
] + ["Others"]

cross["topic_group"] = pd.Categorical(
    cross["topic_group"],
    categories=TOPIC_ORDER,
    ordered=True
)

print(f"LCC users in cross-layer analysis : {cross['user'].nunique():,}")
print(f"User-topic edges                 : {len(cross):,}")
display(
    cross["topic_group"]
    .value_counts(sort=False)
    .rename("n_user_topic_edges")
    .to_frame()
)


### 14.1 — Overall SNA community × semantic-topic association

In [ ]:

overall_table = pd.crosstab(
    cross["sna_community"],
    cross["topic_group"],
    dropna=False
).reindex(columns=TOPIC_ORDER, fill_value=0)

overall_table = overall_table.loc[
    overall_table.sum(axis=1) > 0,
    overall_table.sum(axis=0) > 0
]

chi2_all, p_all, dof_all, expected_all = chi2_contingency(
    overall_table
)
n_all = overall_table.to_numpy().sum()
v_all = np.sqrt(
    chi2_all
    / (n_all * (min(overall_table.shape) - 1))
)

print("All SNA communities × Top-5 topics + Others")
print(f"Chi² = {chi2_all:.3f}")
print(f"df   = {dof_all}")
print(f"p    = {p_all:.3e}")
print(f"Cramér's V = {v_all:.3f}")

pd.DataFrame([{
    "chi2": chi2_all,
    "df": dof_all,
    "p_value": p_all,
    "cramers_v": v_all,
    "n_user_topic_edges": n_all,
    "n_sna_communities": overall_table.shape[0],
    "n_topic_groups": overall_table.shape[1],
}]).to_csv(
    os.path.join(SAVE_DIR, "cross_network_overall_chi2_cramers_v.csv"),
    index=False
)


### 14.2 — Community-specific goodness-of-fit + BH-FDR + Cohen's w

In [ ]:

platform_dist = (
    cross["topic_group"]
    .value_counts(normalize=True, sort=False)
    .reindex(TOPIC_ORDER)
)

community_results = []

for comm, grp in cross.groupby("sna_community"):
    obs = (
        grp["topic_group"]
        .value_counts(sort=False)
        .reindex(TOPIC_ORDER, fill_value=0)
        .astype(float)
    )
    exp = platform_dist * len(grp)

    chi2_gof, p_gof = chisquare(obs, exp)

    # Cohen's w for a chi-square goodness-of-fit test:
    # w = sqrt(chi-square / N)
    cohen_w = np.sqrt(chi2_gof / len(grp))

    top3 = (
        grp["topic_group"]
        .value_counts()
        .head(3)
        .index
        .astype(str)
        .tolist()
    )

    community_results.append({
        "community": int(comm),
        "n_users": grp["user"].nunique(),
        "n_user_topic_edges": len(grp),
        "chi2": chi2_gof,
        "p_raw": p_gof,
        "cohen_w": cohen_w,
        "top1": top3[0] if len(top3) > 0 else "",
        "top2": top3[1] if len(top3) > 1 else "",
        "top3": top3[2] if len(top3) > 2 else "",
    })

community_gof_df = pd.DataFrame(community_results)

_, p_fdr, _, _ = multipletests(
    community_gof_df["p_raw"],
    method="fdr_bh"
)
community_gof_df["p_fdr"] = p_fdr
community_gof_df["significant_fdr_0_05"] = (
    community_gof_df["p_fdr"] < 0.05
)

community_gof_df = community_gof_df.sort_values(
    ["n_users", "p_fdr"],
    ascending=[False, True]
)

print("SNA communities tested:", len(community_gof_df))
print(
    "FDR-significant:",
    int(community_gof_df["significant_fdr_0_05"].sum())
)
display(community_gof_df.head(20))

community_gof_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "community_topic_goodness_of_fit_FDR_effect_size.csv"
    ),
    index=False
)


### 14.3 — Adjusted-standardized-residual heatmap

The original code used `(O-E)/sqrt(E)`, which is a Pearson residual while the figure
was labeled “standardized residual.” This revised plot uses **adjusted standardized
residuals** and displays only Top-5 topics + Others.


In [ ]:

top10_sna = (
    user_community_df["community"]
    .value_counts()
    .head(10)
    .index
    .astype(int)
    .tolist()
)

plot_cross = cross.loc[
    cross["sna_community"].isin(top10_sna)
].copy()

plot_table = pd.crosstab(
    plot_cross["sna_community"],
    plot_cross["topic_group"],
    dropna=False
).reindex(
    index=top10_sna,
    columns=TOPIC_ORDER,
    fill_value=0
)

chi2_top10, p_top10, dof_top10, expected_top10 = chi2_contingency(
    plot_table
)

obs = plot_table.to_numpy(dtype=float)
expected = np.asarray(expected_top10, dtype=float)
n_top10 = obs.sum()

row_prop = obs.sum(axis=1, keepdims=True) / n_top10
col_prop = obs.sum(axis=0, keepdims=True) / n_top10

denom = np.sqrt(
    expected * (1 - row_prop) * (1 - col_prop)
)

adj_resid = np.divide(
    obs - expected,
    denom,
    out=np.zeros_like(obs),
    where=denom > 0
)

v_top10 = np.sqrt(
    chi2_top10
    / (n_top10 * (min(plot_table.shape) - 1))
)

community_user_sizes = (
    user_community_df["community"]
    .value_counts()
)

resid_df = pd.DataFrame(
    adj_resid,
    index=[
        f"Community {c} (n={int(community_user_sizes.loc[c])})"
        for c in plot_table.index
    ],
    columns=plot_table.columns
)

# Annotate only cells with |adjusted residual| >= 2.
annot = resid_df.copy().astype(object)
for r in range(resid_df.shape[0]):
    for c in range(resid_df.shape[1]):
        val = resid_df.iloc[r, c]
        annot.iloc[r, c] = f"{val:.1f}" if abs(val) >= 2 else ""

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    resid_df,
    annot=annot,
    fmt="",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    linecolor="0.85",
    cbar_kws={"label": "Adjusted standardized residual"},
    ax=ax
)

ax.set_title(
    "SNA Community × Semantic Topic: Adjusted Standardized Residuals\n"
    f"Top 10 SNA communities; χ²={chi2_top10:.1f}, "
    f"df={dof_top10}, p={p_top10:.2e}, Cramér's V={v_top10:.3f}"
)
ax.set_xlabel("Semantic topic")
ax.set_ylabel("SNA community")
ax.tick_params(axis="x", rotation=25)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()

resid_fig_path = os.path.join(
    SAVE_DIR,
    "cross_network_adjusted_residuals_top10_TOP5_PLUS_OTHERS.png"
)
plt.savefig(
    resid_fig_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

resid_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "cross_network_adjusted_standardized_residuals_top10.csv"
    )
)

sig_cells = (
    resid_df
    .where(resid_df.abs() >= 2)
    .stack()
    .reset_index()
)
sig_cells.columns = [
    "sna_community",
    "semantic_topic",
    "adjusted_standardized_residual"
]
sig_cells["direction"] = np.where(
    sig_cells["adjusted_standardized_residual"] > 0,
    "overrepresented",
    "underrepresented"
)

sig_cells.to_csv(
    os.path.join(
        SAVE_DIR,
        "cross_network_residual_cells_abs_ge_2.csv"
    ),
    index=False
)

print("Saved:", resid_fig_path)
print(
    f"Top-10 test: Chi²={chi2_top10:.1f}, df={dof_top10}, "
    f"p={p_top10:.3e}, Cramér's V={v_top10:.3f}"
)
display(
    sig_cells.sort_values(
        "adjusted_standardized_residual",
        key=lambda x: x.abs(),
        ascending=False
    ).head(30)
)


## 15 — Downstream completion checklist

In [ ]:

expected_outputs = [
    "final_top5_topic_labels.csv",
    "topic_size_by_user_participation.csv",
    "user_topic_bridging_summary.csv",
    "user_topic_participation_breadth.csv",
    "topic_topic_user_overlap.csv",
    "user_topic_coverage_legacy_vs_corrected.csv",
    "corrected_user_topic_status_all_users.csv",
    "corrected_no_topic_users_audit.csv",
    "corrected_no_topic_reason_summary.csv",
    "topic_vs_no_topic_user_characteristics.csv",
    "topic_vs_no_topic_interaction_distribution.csv",
    "topic_vs_no_topic_keyword_distribution.csv",
    "topic_vs_no_topic_top_terms.csv",
    "no_topic_post_sample_for_manual_review.csv",
    "user_topic_bipartite_top5_ORIGINAL_STYLE_FIXED.png",
    "community_type_counts_row_level_fixed.csv",
    "topic_by_community_type_counts_fixed.csv",
    "topic_by_community_type_proportions_fixed.csv",
    "top5_topic_by_community_type_proportions_labeled.csv",
    "interaction_type_community_relation_chi2_effect_size.csv",
    "threshold_sensitivity_summary_manuscript.csv",
    "threshold_sensitivity_dualaxis_MANUSCRIPT.png",
    "cross_network_overall_chi2_cramers_v.csv",
    "community_topic_goodness_of_fit_FDR_effect_size.csv",
    "cross_network_adjusted_residuals_top10_TOP5_PLUS_OTHERS.png",
    "cross_network_adjusted_standardized_residuals_top10.csv",
    "cross_network_residual_cells_abs_ge_2.csv",
]

missing_outputs = [
    x for x in expected_outputs
    if not os.path.exists(os.path.join(SAVE_DIR, x))
]

if missing_outputs:
    print("WARNING — missing downstream output(s):")
    for x in missing_outputs:
        print(" -", x)
else:
    print("✅ 02 completed: all expected downstream outputs were created.")

print("\nOutput folder:", SAVE_DIR)
